# Online Retail — RFM Scoring & Customer Segmentation

**Author:** Khaled Waleed  
**Dataset:** `retail_clean.csv` — output from Phase 1 EDA  
**Tool stack:** Python · Pandas · NumPy · Matplotlib · Seaborn

---

## What is RFM?

RFM is a behaviour-based framework that scores every customer on three dimensions:

| Dimension | Question it answers | How it's measured |
|-----------|--------------------|-----------------|
| **Recency (R)** | How recently did this customer buy? | Days since last purchase |
| **Frequency (F)** | How often do they buy? | Number of unique orders |
| **Monetary (M)** | How much do they spend? | Total revenue generated |

Each dimension is scored **1–5** (5 = best). Combining the three scores gives us a single RFM profile per customer that we use to assign them to a business segment.

---

## Notebook structure

1. [Load clean data](#1-load-clean-data)  
2. [Calculate RFM values](#2-calculate-rfm-values)  
3. [Score & rank each dimension](#3-score--rank-each-dimension)  
4. [Assign customer segments](#4-assign-customer-segments)  
5. [Segment analysis & visualisation](#5-segment-analysis--visualisation)  
6. [Export](#6-export)

---

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ── Global plot style ────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi'       : 150,
    'axes.titlesize'   : 14,
    'axes.labelsize'   : 12,
    'axes.titleweight' : 'bold',
    'figure.facecolor' : 'white',
})

# ── Colour palette (consistent with Phase 1) ─────────────────
TEAL   = '#1D9E75'
CORAL  = '#D85A30'
PURPLE = '#534AB7'
AMBER  = '#BA7517'
GRAY   = '#888780'

# Segment colours — used consistently in every chart
SEGMENT_COLORS = {
    'Champions'       : '#1D9E75',   # teal
    'Loyal'           : '#534AB7',   # purple
    'Potential Loyal' : '#3B8BD4',   # blue
    'At Risk'         : '#BA7517',   # amber
    'Lost'            : '#D85A30',   # coral
}

---
## 1. Load clean data

We pick up exactly where Phase 1 left off — the clean, enriched CSV.

In [ ]:
df = pd.read_csv('retail_clean.csv', parse_dates=['InvoiceDate'])

print(f'Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Period : {df["InvoiceDate"].min().date()}  →  {df["InvoiceDate"].max().date()}')
print(f'Customers : {df["Customer ID"].nunique():,}')
df.head(3)

---
## 2. Calculate RFM values

We define a **snapshot date** — the day after the last transaction in the dataset.  
This is the reference point for Recency: the more days since the last purchase, the less recent the customer.

> Using `max_date + 1 day` is a common convention so that the most active customer
> gets a Recency of 1 (not 0), which avoids issues when scoring on a 1–5 scale.

In [ ]:
# Snapshot date: one day after the last transaction in the dataset
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

print(f'Snapshot date (reference for Recency): {snapshot_date.date()}')

In [ ]:
# Aggregate one row per customer with the three RFM metrics
rfm = (
    df.groupby('Customer ID')
    .agg(
        Recency   = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
        Frequency = ('Invoice',     'nunique'),   # unique orders, not line items
        Monetary  = ('TotalPrice',  'sum')
    )
    .reset_index()
)

# Round Monetary to 2 decimal places
rfm['Monetary'] = rfm['Monetary'].round(2)

print(f'RFM table: {len(rfm):,} customers')
rfm.head()

In [ ]:
# Distribution summary — helps us understand the spread before scoring
rfm[['Recency', 'Frequency', 'Monetary']].describe().round(2)

**What to notice here:**
- **Recency** has a wide range — some customers bought very recently, others over a year ago.
- **Frequency** is heavily skewed — the median is likely low (1–2 orders) but the max can be very high; classic retail pattern.
- **Monetary** will also be right-skewed — a small group of high-spenders inflates the mean.

---
## 3. Score & rank each dimension

We convert the raw R / F / M values into **scores from 1 to 5** using quintile-based binning (`pd.qcut`).

**Scoring direction matters:**
- For **Recency** — a *lower* number of days is better, so we reverse the labels (1 = bought long ago, 5 = bought very recently).
- For **Frequency** and **Monetary** — a *higher* value is better, so labels go 1 → 5 naturally.

In [ ]:
# qcut divides customers into 5 equal-sized groups (quintiles)
# duplicates='drop' handles ties at bin edges gracefully

rfm['R_score'] = pd.qcut(rfm['Recency'],   q=5, labels=[5, 4, 3, 2, 1], duplicates='drop')
rfm['F_score'] = pd.qcut(rfm['Frequency'], q=5, labels=[1, 2, 3, 4, 5], duplicates='drop')
rfm['M_score'] = pd.qcut(rfm['Monetary'],  q=5, labels=[1, 2, 3, 4, 5], duplicates='drop')

# Convert to integer for arithmetic
rfm[['R_score', 'F_score', 'M_score']] = rfm[['R_score', 'F_score', 'M_score']].astype(int)

# Combined RFM score (simple sum: range 3–15)
rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

print('Score distribution:')
rfm[['R_score', 'F_score', 'M_score', 'RFM_score']].describe().round(2)

---
## 4. Assign customer segments

We map R and F scores to five business segments using a rule-based approach.  
Monetary is captured in the RFM score but R and F are the primary drivers of segment definition —  
recency tells us *if* a customer is still engaged, frequency tells us *how deeply*.

| Segment | R score | F score | Business meaning |
|---------|---------|---------|------------------|
| **Champions** | 4–5 | 4–5 | Bought recently and often — your best customers |
| **Loyal** | 2–5 | 3–5 | Buy regularly but not always the most recent |
| **Potential Loyal** | 3–5 | 1–3 | Recent buyers who haven't established a habit yet |
| **At Risk** | 2–3 | 2–5 | Used to buy but haven't recently — need re-engagement |
| **Lost** | 1–2 | 1–2 | Low recency and low frequency — likely churned |

In [ ]:
def assign_segment(row):
    """Map R_score and F_score to a customer segment label."""
    r, f = row['R_score'], row['F_score']

    if r >= 4 and f >= 4:
        return 'Champions'
    elif f >= 3 and r >= 2:
        return 'Loyal'
    elif r >= 3 and f <= 3:
        return 'Potential Loyal'
    elif r <= 3 and f >= 2:
        return 'At Risk'
    else:
        return 'Lost'


rfm['Segment'] = rfm.apply(assign_segment, axis=1)

# Quick count per segment
segment_counts = rfm['Segment'].value_counts()
print('Customers per segment:')
print(segment_counts.to_string())

---
## 5. Segment analysis & visualisation

Four charts that tell the full story of the customer base:

- **5-A** Customer count per segment (who's in each group?)
- **5-B** Revenue contribution per segment (which group drives the most value?)
- **5-C** Avg RFM metrics per segment (what makes each group different?)
- **5-D** Recency vs Frequency scatter (where do customers sit in the 2D space?)

### 5-A · Customer count per segment

In [ ]:
seg_order  = segment_counts.index.tolist()
seg_colors = [SEGMENT_COLORS[s] for s in seg_order]

fig, ax = plt.subplots(figsize=(9, 4))

bars = ax.bar(seg_order, segment_counts.values, color=seg_colors, edgecolor='white', linewidth=0.5)

# Show count and percentage on each bar
total_customers = len(rfm)
for bar, count in zip(bars, segment_counts.values):
    pct = count / total_customers * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 15,
        f'{count:,}\n({pct:.1f}%)',
        ha='center', va='bottom', fontsize=9
    )

ax.set_title('Customer Count by Segment')
ax.set_ylabel('Number of Customers')
ax.set_xlabel('')

plt.tight_layout()
plt.savefig('06_segment_counts.png')
plt.show()

### 5-B · Revenue contribution per segment

Customer count tells us *who's there* — revenue tells us *who matters most to the business*.

In [ ]:
seg_revenue = (
    rfm.groupby('Segment')['Monetary']
    .sum()
    .reindex(seg_order)  # keep the same segment order as the count chart
)

fig, ax = plt.subplots(figsize=(9, 4))

bars = ax.bar(seg_revenue.index, seg_revenue.values, color=seg_colors, edgecolor='white', linewidth=0.5)

total_revenue = rfm['Monetary'].sum()
for bar, val in zip(bars, seg_revenue.values):
    pct = val / total_revenue * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + total_revenue * 0.005,
        f'£{val/1000:.0f}K\n({pct:.1f}%)',
        ha='center', va='bottom', fontsize=9
    )

ax.set_title('Total Revenue by Segment')
ax.set_ylabel('Revenue (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('07_segment_revenue.png')
plt.show()

### 5-C · Average RFM metrics per segment

A heatmap makes it easy to see what makes each segment distinct across all three dimensions at once.

In [ ]:
seg_profile = (
    rfm.groupby('Segment')[['Recency', 'Frequency', 'Monetary']]
    .mean()
    .round(1)
    .reindex(seg_order)
)

# Normalise each column to 0–1 so the heatmap isn't dominated by Monetary scale
seg_profile_norm = (seg_profile - seg_profile.min()) / (seg_profile.max() - seg_profile.min())

fig, ax = plt.subplots(figsize=(8, 4))

sns.heatmap(
    seg_profile_norm,
    annot=seg_profile,       # show the real (un-normalised) values as labels
    fmt='g',
    cmap='YlGn',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Normalised score'}
)

ax.set_title('Average RFM Profile per Segment')
ax.set_xlabel('')
ax.set_ylabel('')

plt.tight_layout()
plt.savefig('08_rfm_heatmap.png')
plt.show()

### 5-D · Recency vs Frequency scatter

A scatter plot in the R × F space shows the natural clustering of customers — each segment should occupy a distinct region.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for segment, group in rfm.groupby('Segment'):
    ax.scatter(
        group['Recency'],
        group['Frequency'],
        label=segment,
        color=SEGMENT_COLORS[segment],
        alpha=0.55,
        edgecolors='white',
        linewidth=0.3,
        s=40
    )

ax.set_title('Customer Segments — Recency vs Frequency')
ax.set_xlabel('Recency (days since last purchase)  ←  more recent')
ax.set_ylabel('Frequency (unique orders)')
ax.legend(title='Segment', bbox_to_anchor=(1.01, 1), loc='upper left')

# Invert x-axis so "more recent" customers appear on the right
ax.invert_xaxis()

plt.tight_layout()
plt.savefig('09_scatter_r_vs_f.png')
plt.show()

**Insight:** Champions cluster in the **top-right** (recent, frequent buyers). Lost customers appear in the **bottom-left** (old, infrequent). The scatter plot also reveals how spread the Potential Loyal and At Risk groups are — a useful signal that those segments may benefit from further sub-segmentation in future iterations.

### Segment summary table

A consolidated view of each segment's size and value — useful to paste directly into a report or Power BI text card.

In [ ]:
summary = (
    rfm.groupby('Segment')
    .agg(
        Customers      = ('Customer ID',  'count'),
        Avg_Recency    = ('Recency',      'mean'),
        Avg_Frequency  = ('Frequency',    'mean'),
        Avg_Monetary   = ('Monetary',     'mean'),
        Total_Revenue  = ('Monetary',     'sum'),
    )
    .round(1)
    .reindex(seg_order)
)

summary['Revenue_Share_%'] = (summary['Total_Revenue'] / summary['Total_Revenue'].sum() * 100).round(1)

summary

---
## 6. Export

We export two files:
- **`rfm_scored.csv`** — full customer-level RFM table (input for Power BI dashboard)
- **`rfm_segments_summary.csv`** — aggregated segment summary (useful for Power BI KPI cards)

In [ ]:
RFM_PATH     = 'rfm_scored.csv'
SUMMARY_PATH = 'rfm_segments_summary.csv'

rfm.to_csv(RFM_PATH, index=False)
summary.to_csv(SUMMARY_PATH)

print(f'Saved → {RFM_PATH}        ({len(rfm):,} rows)')
print(f'Saved → {SUMMARY_PATH}  ({len(summary)} rows)')

In [ ]:
# Final sanity check — preview the scored table
print('── rfm_scored.csv preview ──')
rfm.sample(8, random_state=42)

---
## Next Steps

**Phase 3 — Power BI Dashboard**

Load `rfm_scored.csv` into Power BI and build the three-page dashboard:

| Page | Content |
|------|---------|
| **1 — Executive Overview** | Total Revenue, Orders, Avg Order Value, Monthly trend line, Top countries map |
| **2 — RFM Segments** | Segment donut, Revenue by segment bar, R × F scatter, Segment slicer |
| **3 — Product Analysis** | Top 10 products, Returns rate, Qty vs Revenue, Monthly heatmap |

**Key DAX measures to create in Power BI:**
```
Total Revenue  = SUM(rfm_scored[Monetary])
Avg Recency    = AVERAGE(rfm_scored[Recency])
Champions %    = DIVIDE(COUNTROWS(FILTER(rfm_scored, rfm_scored[Segment] = "Champions")), COUNTROWS(rfm_scored))
```